# CS410 Homework 6: Linear Regression & Model Selection

## Load Necessary Imports 

In [ ]:
from typing import Optional, Any, Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.base import BaseEstimator
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

from regression_support import prepare_cv_data, make_regression_pipeline, plot_cv_errors


<img src="https://hackmd.io/_uploads/BysoYdd_zx.jpg" alt="crossy road linear regression" width="600">

## Part 1: Data Cleaning & Preprocessing

Throughout the season, Chef Chicken has worked tirelessly to guide her flock across the highway, all while making sure her children are well fed with plenty of delicious corn. Now that the run has reached the eight-lane stretch, her focus has shifted to one critical goal: keeping the flock alive. Aware that the truck lane cycles faster than anything he has crossed before, and that there isn't a single log or lily pad in sight, Chef Chicken is eager to estimate her flock's chances of survival.

In her quest for answers, Chef Chicken discovered some survival data collected by the World Hen Organization, *a.k.a.* the [World Health Organization (WHO)](https://www.who.int/). Since she is still new to the art of traffic dodging, she's hoping you can use these data to help her predict the life expectancy of the birds braving the road. With the traffic closing in, she is counting on you to help her stay one hop ahead and get the whole flock to the other side!

### Data

The `life_expectancy.csv` file is in the same folder as this notebook. This dataset contains health indicators for a variety of countries--ranging from Afghanistan to Zimbabwe--between the years 2000 and 2015. It contains statistics on life expectancy for each country, related health categories like infant deaths, incidence of measles, and BMI, as well as country-specific data like GDP and population size. In addition, it labels countries as developing or developed. The WHO's intent in collecting these data is to empower governments to improve life expectancy by identifying and then addressing the most impactful factors.

The original dataset is located [here](https://www.kaggle.com/datasets/kumarajarshi/life-expectancy-who?resource=download). We have provided this dataset for you, but it still needs to be cleaned—a necessary step before embarking upon any data science or machine learning task. You will clean it in two steps: first **min-max scale** the numeric features using the original dataset, and then dropping all rows with missing values. To min-max scale a feature, you subtract its minimum value and then divide by its range (i.e., the difference between its maximum and its minimum), so that every feature value falls in the range $[0, 1]$.

> **Tip:** Descriptions of the features are located in `features.txt`. Read through this file, and `life_expectancy.csv` as well, so that you can later interpret the models you build.

#### Task 1.1: Data Cleaning (AI)

> Prompt a generative AI to write a function that:
> - leaves `Country` and `Status` unchanged
> - **min-max scales** every numeric feature except `life_expectancy`, using each column's minimum and maximum from the original dataset
> - makes the unscaled `life_expectancy` column the rightmost column in the dataset
> - drops all rows containing missing values
>
> Run the code the AI generates, expand "Correctness check:" below to check your result against ours, and then prompt the AI to fix the code if it is wrong.
>
> **Hint:** Be sure to compute scaling bounds *before* removing rows. Otherwise, your solution will not match ours.
>
> **Note:** As always, you are responsible for the code you submit. Make sure you understand the code the generative AI produces. Record your prompts in your README under "AI Use Description" and submit your chat transcript.
>
> **Tip:** If you cannot get your version to work, you can replace `preprocess_data(pd.read_csv('life_expectancy.csv'))` with `pd.read_csv('life_expectancy_preprocessed.csv')` to use our pre-processed copy of the data (also stored in this folder) instead of your own in the later parts of this assignment.

In [ ]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    First min-max scale numeric features using bounds from the original data,
    then drop rows with missing values. Leave non-numeric columns unchanged
    and move the unscaled life_expectancy column to the end.

    - Inputs:
        - df: DataFrame containing the data
    - Outputs:
        - data: DataFrame with min-max scaled numeric columns and dropped missing values
    """
    # TODO (AI-Assisted):
    pass

In [ ]:
processed = preprocess_data(pd.read_csv('life_expectancy.csv'))
print("Shape:", processed.shape)
processed.head()

<details><summary><b>Correctness check:</b></summary>

With our implementation, the preview cell above prints shape **(1649, 22)** -- all rows with missing values are removed, every numeric feature lies in $[0, 1]$, the non-numeric columns (`Country`, `Status`) are unchanged, and `life_expectancy` is unscaled as the rightmost column.

```text
       Country      Year      Status  Adult_Mortality  infant_deaths  Alcohol  percentage expenditure  Hepatitis_B   Measles       BMI  under_five_deaths     Polio  Total_expenditure  Diphtheria  HIV/AIDS       GDP  Population  thinness_1-19_years  thinness_5-9_years  Income_composition_of_resources  Schooling  life_expectancy
0  Afghanistan  1.000000  Developing         0.362881       0.034444      0.0                0.003659     0.653061  0.005439  0.209733             0.0332  0.031250           0.452118    0.649485       0.0  0.004889    0.026074             0.619565            0.603509                         0.505274   0.487923             65.0
1  Afghanistan  0.933333  Developing         0.373961       0.035556      0.0                0.003774     0.622449  0.002319  0.203940             0.0344  0.572917           0.453279    0.618557       0.0  0.005127    0.000253             0.630435            0.610526                         0.502110   0.483092             59.9
2  Afghanistan  0.866667  Developing         0.369806       0.036667      0.0                0.003759     0.642857  0.002027  0.198146             0.0356  0.614583           0.450377    0.639175       0.0  0.005287    0.024525             0.637681            0.617544                         0.495781   0.478261             59.9
3  Afghanistan  0.800000  Developing         0.375346       0.038333      0.0                0.004014     0.673469  0.013135  0.192352             0.0372  0.666667           0.473012    0.670103       0.0  0.005608    0.002857             0.644928            0.628070                         0.488397   0.473430             59.5
4  Afghanistan  0.733333  Developing         0.379501       0.039444      0.0                0.000364     0.683673  0.014200  0.187717             0.0388  0.677083           0.435287    0.680412       0.0  0.000519    0.002302             0.655797            0.635088                         0.478903   0.458937             59.2
```

</details>

### Correlation

Correlation is a statistical measure between two random variables that describes their linear relationship to one another. Its values range from $-1$ to $+1$, with negative values suggesting a negative correlation (one increases as the other decreases), and positive values suggesting a positive correlation (the two variables move in the same direction).

Correlation is closely related to covariance. Recall the definition of covariance from last week's assignment. The **covariance** of two random variables $Y$ and $Z$ is given by:

$$
\textrm{Cov}(Y, Z) = \mathbb{E} \left[ (Y - \mathbb{E} \left[ Y \right]) (Z - \mathbb{E} \left[ Z \right]) \right]
$$

**Correlation** is the covariance of two random variables, divided by the product of their standard deviations. **Standard deviation** is the square root of variance. Therefore,

$$
\textrm{Corr}(Y, Z) = \frac{\textrm{Cov}(Y, Z)}{\sqrt{\textrm{Var}(Y)} \sqrt{\textrm{Var}(Z)}}
$$

Dividing by the standard deviations of $Y$ and $Z$ serves to standardize covariance values, so that correlation always lies in the range $[-1, 1]$.

A **sample correlation**, like a sample mean, is calculated from data. The **sample mean** $\bar{x}$ of a random vector $X \in \mathbb{R}^d$, assuming $m$ samples $x_k \in \mathbb{R}^d$ of $X$ with $k \in \{1, \ldots, m\}$, is defined as $\frac{1}{m} \sum_{k=1}^m x_k$. The sample mean of the `life_expectancy` feature, for example, is the average of all the `life_expectancy` values. The formula for the sample correlation between features $Y$ and $Z$, again given $m$ samples of each, is as follows:

$$
r = \frac{\sum_{k=1}^m \left( y_{k} - \bar{y} \right) \left( z_{k} - \bar{z} \right)}{\sqrt{\sum_{k=1}^m \left( x_{k} - \bar{x} \right)^2} \sqrt{\sum_{k=1}^m \left( y_{k} - \bar{y} \right)^2}}
$$


A **correlation matrix** $R$ describes the correlations between all $d$ features in a dataset $X \in \mathbb{R}^{m \times d}$. In particular, $r_{ij}$ is the sample correlation between feature $i$ and feature $j$, for $i, j \in \{ 1, \ldots, d \}$, e.g., the correlation between `Schooling` and `life_expectancy`, or `Schooling` and `infant_deaths`. The formula for the sample correlation matrix is as follows:

$$
r_{ij} = \frac{\sum_{k=1}^m \left( x_{ki} - \bar{x}_i \right) \left( x_{kj} - \bar{x}_j \right)}{\sqrt{\sum_{k=1}^m \left( x_{ki} - \bar{x}_i \right)^2} \sqrt{\sum_{k=1}^m \left( x_{kj} - \bar{x}_j \right)^2}}
$$

Intuitively, the data that inhabit each column $k$ of the matrix $X$ represent $m$ samples of an underlying random variable $X_k$ (e.g., `HIV/AIDS`), and the sample correlation $r_{ij}$ is the sample correlation between $X_i$ and $X_j$.

Your next task is to implement the **sample correlation matrix** $R$ for the `life_expectancy.csv` dataset. You should do so with the help of generative AI.

#### Task 1.2: Data Preprocessing (AI)

> Given the dataset $X$ with corresponding feature names, the code snippet below is intended to
>
> - compute the correlation matrix $R$ for the dataset $X$
> - print the 10 most positively and 10 most negatively correlated pairs of features
> - plot $R$ as a heatmap with the features labeled on both axes and an accompanying color bar
>
> Prompt a generative AI to fill in the snippet.
>
> Run the code the AI generates, expand "Correctness check:" below to check your result against ours, and then prompt the AI to fix the code if it is wrong.
>
> **Extra Fun:** Prompt the AI to reorganize the variables in the heatmap so that it reads like a smooth gradient, placing highly correlated variables near one another. **Hint:** Ask the AI to compute the first principal component of $R$ and sort the variables accordingly.
>
> **Note:** As always, you are responsible for the code you submit. Make sure you understand the code the generative AI produces (except perhaps the PCA analysis). Record your prompts in your README under "AI Use Description" and submit your chat transcript.

In [ ]:
def corr(X: np.ndarray, feature_names: List[str]) -> None:
    """
    Calculate the correlation matrix for the given dataset.
    X is a n x d matrix, where n is the number of samples and d is the number of features.
    feature_names is a list of feature names (length d) corresponding to the columns of X.

    - Inputs:
        - X: Features (np.ndarray)
        - feature_names: Names of the features (list of strings)
    """
    # TODO (AI-Assisted): 
    pass

In [ ]:
# correlation over the full dataset (all country-year rows), original column order
numeric = pd.read_csv('life_expectancy.csv').select_dtypes(include='number')
corr(numeric.to_numpy(dtype=float), numeric.columns)

<details><summary><b>Correctness check:</b></summary>

Your plot does not need to be pixel-identical to ours, but the correlation values, and therefore the pattern of the colors and the printed pairs, should match:

![correlation_matrix](figs/expected/correlation_matrix.png)

```text
=== Top 10 Positively Correlated Feature Pairs ===
infant_deaths and under_five_deaths: 0.9966
thinness_1-19_years and thinness_5-9_years: 0.9391
percentage expenditure and GDP: 0.8994
Income_composition_of_resources and Schooling: 0.8001
life_expectancy and Schooling: 0.7520
life_expectancy and Income_composition_of_resources: 0.7248
Polio and Diphtheria: 0.6736
Hepatitis_B and Diphtheria: 0.6115
life_expectancy and BMI: 0.5677
infant_deaths and Population: 0.5568

=== Top 10 Negatively Correlated Feature Pairs ===
life_expectancy and Adult_Mortality: -0.6964
life_expectancy and HIV/AIDS: -0.5566
BMI and thinness_5-9_years: -0.5389
BMI and thinness_1-19_years: -0.5320
life_expectancy and thinness_1-19_years: -0.4772
thinness_1-19_years and Schooling: -0.4717
life_expectancy and thinness_5-9_years: -0.4716
thinness_5-9_years and Schooling: -0.4606
Adult_Mortality and Income_composition_of_resources: -0.4576
Adult_Mortality and Schooling: -0.4546
```

</details>

#### Task 1.3: MCQs

> **Q1.** A feature has a correlation coefficient of **-0.85** with life expectancy. Which of the following statements is the most accurate?
>
> &emsp;**A.** The feature is weakly related to life expectancy.
>
> &emsp;**B.** As the feature increases, life expectancy tends to strongly decrease, but this alone does not establish causation.
>
> &emsp;**C.** As the feature increases, life expectancy tends to strongly increase, but this alone does not establish causation.
>
> &emsp;**D.** A negative coefficient means the correlation is unreliable, as only positive values indicate a correlation.
>
>
> **Q2.** After running the correlation analysis on the life-expectancy dataset, among the four features listed below, which shows the strongest **positive** correlation with life expectancy?
>
> &emsp;**A.** Measles
>
> &emsp;**B.** HIV/AIDS
>
> &emsp;**C.** Adult mortality
>
> &emsp;**D.** Income composition of resources
>
>
> **Q3.** Which feature shows the strongest **negative** correlation?
>
> &emsp;**A.** Adult mortality
>
> &emsp;**B.** Schooling
>
> &emsp;**C.** BMI
>
> &emsp;**D.** Alcohol

## Part 2: Simple Linear Regression

In this part of the assignment, you will be building several **simple** linear regression models, meaning linear regression models based on just one feature. The primary challenge is to choose sufficiently explanatory variables.

### Task 2.1: Predict & Fit

> Implement the closed-form solution to least-squares optimization in the `fit` method in the `LinearRegression` class in the cell below. Then implement `predict` to make a prediction based on the output of `fit`.

<details><summary><b>Hint: Incorporating Bias</b></summary>

An advantage to running a simple regression is that the dimensionality of $X$ is $m \times 1$, i.e., $X$ is just a vector, say $x \in \mathbb{R}^m$, so that $X^T X = x^T x$ is a scalar, meaning $X^T X$ is invertible (as long as $X \ne 0$, which would not make for a very interesting explanatory variable!).

To incorporate bias, you should prepend a vector of 1s to the matrix $X$ before computing the closed-form solution. Note that $X^T X$ likely remains invertible even with an additional parameter, since what is required for $X^T X$ to be invertible is that the columns of $X$ be linearly independent. Just as a column of 0's would not make for a very interesting explanatory variable, nor would a constant-valued column.

</details>
 
### Task 2.2: Mean-Squared Error

>How good of a fit is the model you just built?
>
> Given a vector of predictions $\hat{y} \in \mathbb{R}^n$ and a vector of observations $y \in \mathbb{R}^n$, the **mean-squared error (MSE)** between $\hat{y}$ and $y$ is defined as follows:
>
$$
\textrm{MSE} (\hat{y}, y) = \frac{1}{n} \sum_{i = 1}^n \left( \hat{y}_i - y_i \right)^2
$$

> Implement the `mse` method in the `LinearRegression` class to compute the mean-squared error between a vector of predictions $X w$ and a vector of labels $y$.

<details><summary><b>Hint: Signature</b></summary>

The `mse` function takes as input a matrix $X$ and a vector $y$, computes $\hat{y} = X w$, and then outputs $\textrm{MSE} (\hat{y}, y)$.

</details>

### Task 2.3: Simple Linear Regression

> Choose three features that are highly correlated–either positively or negatively–with life expectancy based on your correlation analysis above, and regress `life_expectancy` on each one in `single_feature_regression` using your `fit` function. Then use your `mse` function to calculate the mean-squared error for each model, both on a training set and a test set.
>
> Use the provided `extract_features` helper in `regression_support.py`, imported in the implementation cell below, which returns the requested feature and the target, `life_expectancy`, after eliminating duplicates; in particular, it only returns one observation per country.
>
> In your README, summarize your three simple linear regression models, reporting the training and test error in all cases. Which one seemed to discover the best fit visually? Did this one also yield the lowest training and test MSE?

<details><summary><b>Hint: Plots</b></summary>

Use the `plot_model` method provided by LinearRegression to visualize the three linear models.

</details>

In [ ]:
from regression_support import extract_features

class LinearRegression(BaseEstimator):
    """
    A class that implements linear regression (fit and predict)
    """

    def __init__(self, coefficients: Optional[np.ndarray] = None) -> None:
        self.coefficients = coefficients

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegression':
        """
        Fit a linear regression model to the given data.
        Add a column of ones to X for the intercept term.
        Use the closed-form equation to find the coefficients of the linear regression model.
        Store the coefficients in the coefficients attribute.

        - Input:
            X: a numpy array of shape (n, d) where n is the number of data points and d is the number of features
            y: a numpy array of shape (n,) containing the true labels
        """
        # TODO
        # Add a column of ones to X for the intercept term
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict the labels for the given data using the trained model.
        
        - Input:
            X: a numpy array of shape (n, d) where n is the number of data points and d is the number of features
        - Output:
            y_pred: a numpy array of shape (n,) containing the predicted labels
        """
        # TODO
        pass

    def mse(self, X: np.ndarray, y_true: np.ndarray) -> float:
        """
        Compute the mean squared error of the model on the given data.
        Use predict to compute y_pred and compute the MSE.

        - Input:
            X: a numpy array of shape (n, d) where n is the number of data points and d is the number of features
            y_true: a numpy array of shape (n,) containing the true labels
        - Output:
            The mean squared error of the model on the given data.
        """
        # TODO
        pass

    def get_coefficients(self) -> Optional[np.ndarray]:
        """
        Return the coefficients of the linear regression model.
        """
        return self.coefficients
    
    def plot_model(self, X: np.ndarray, y: np.ndarray, title: str = "", xlabel: str = "", ylabel: str = "") -> None:
        """
        Plot the data points and the model.

        - Input:
            X: a numpy array of shape (n, d) where n is the number of data points and d is the number of features
            y: a numpy array of shape (n,) containing the true labels
            title: a string containing the title of the plot
            xlabel: a string containing the label for the x-axis
            ylabel: a string containing the label for the y-axis
        """
        plt.plot(X, y, 'o')
        xs = np.linspace(X.min(), X.max(), 100)
        ys = self.predict(xs)
        plt.plot(xs, ys, label="Model")
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.show()

    def __sklearn_is_fitted__(self):
        return self.coefficients is not None


def single_feature_regression(X: np.ndarray, y: np.ndarray, feature_name: str) -> None:
    """
    Train a linear regression model for a single feature.
    Plot the model and the data, print train and test MSE.

    - Inputs:
        - X: Features (np.ndarray)
        - y: Target variable (np.ndarray)
        - feature_name: Name of the feature (string)
    """
    # TODO
    pass

# This is just a template using 'Schooling' - feel free to swap in whichever
# feature you'd like to regress on (see the correlation output above for ideas).
df = preprocess_data(pd.read_csv('life_expectancy.csv'))
X, y, feature_names = extract_features(df, 'Schooling')
single_feature_regression(X, y, 'Schooling')


#### Task 2.4: MCQ

> **Q4.** Two models are evaluated on the same dataset, and here are the results:
>
> | Model | Train MSE | Test MSE |
> | ----- | --------- | -------- |
> | A     | 26        | 31       |
> | B     | 8         | 74       |
>
> Which of the following is most accurate?
>
> &emsp;**A.** Model B is the best fit since it has the lowest training error.
>
> &emsp;**B.** Model A is underfitting since its training error is higher than testing error.
>
> &emsp;**C.** Both models generalize equally well since their average errors are similar.
>
> &emsp;**D.** Model B is overfitting since it fits the training data well but generalizes poorly.
>

## Part 3: Ridge Regression and LASSO

Simple linear regression models are built using a single feature, but life expectancy depends on many factors! Your next task is to build multiple regression models, which incorporate multiple features.

Additional features, however, introduce added complexity, so that multiple regression models run the risk of overfitting. You will thus apply regularization to your multiple regression models, to prevent overfitting.

### Prepare the Data

The next cell prepares the data for model building. It

1. Loads the original `life_expectancy.csv` rather than your scaled Part 1 data (or our preprocessed fallback), since scaling will now happen within each model's pipeline.

2. The provided `prepare_cv_data` helper drops rows with missing values and keeps one observation per country.

3. `sklearn`'s `train_test_split` function splits a dataset into a training set and a test set. `X_train` and `y_train` are used for training, while `X_test` and `y_test` are used for testing. By setting `test_size=0.2`, we hold out 20\% of data for testing.

In [ ]:
X, y, feature_names = prepare_cv_data(pd.read_csv('life_expectancy.csv'))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f'Training countries: {len(y_train)}; test countries: {len(y_test)}')


#### Task 3.1: Ridge Regression

Standard linear regression seeks to minimize squared residual error, i.e., $\| X w - y \|_2^2$. **Ridge regression** adds an additional term called a **regularizer** to this objective, namely $\beta \| w \|_2^2$, for some hyperparameter $\beta \ge 0$. The loss function $\mathcal{L}$ for ridge regression is thus given by:

$$
\mathcal{L} = \| X w - y \|_2^2 + \beta \| w \|_2^2
$$

Because the aim is to *minimize* this loss, the regularizer $\beta$ controls how much the model is penalized for using large coefficients.

Just as in the standard linear regression case, we minimize this loss function by solving for $w^*$ such that $\nabla_w \mathcal{L} (w^*) = 0$, which again yields a closed-form solution for $w$, namely:

$$
w = (X^T X +\beta I)^{-1} X^T y
$$

This closed-form solution is a simple generalization of that of linear regression, in which $\beta I$ is added to $X^T X$ before computing the inverse. Indeed, if $\beta = 0$, then we recover the solution to ordinary least squares.

> Implement the `fit` method in `RidgeRegression` in the cell below. As in `LinearRegression`, prepend a column of ones to $X$ (which, by the way, implies that the intercept is also penalized). Since `RidgeRegression` inherits from `LinearRegression`, you do *not* need to implement `predict`.

In [ ]:
class RidgeRegression(LinearRegression):
    def __init__(self, beta: float = 0.1, coefficients: Optional[np.ndarray] = None) -> None:
        self.beta = beta
        self.coefficients = coefficients
  
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'RidgeRegression':
        """
        Fit a ridge regression model to the given data.
        Add a column of ones to X for the intercept term.
        Use the closed-form equation to find the coefficients of the ridge regression model.
        Store the coefficients in the coefficients attribute.

        - Input:
            X: a numpy array of shape (n, d) where n is the number of data points and d is the number of features
            y: a numpy array of shape (n,) containing the true labels
        """
        # TODO
        pass

#### Task 3.1: Ridge Regression (cont'd)

Run the next cell. It calls `make_regression_pipeline`, which chains together all the steps of fitting a model: it selects the requested numeric features, min-max scales them, expands them into polynomial features, standardizes the results, and fits the regressor. It then fits a degree-1 ridge regression model on *all* numeric features. Record this ridge regression model's training MSE.

In [ ]:
ridge_model = make_regression_pipeline(RidgeRegression(beta=0.1))
ridge_model.fit(X_train, y_train)
print('Ridge training MSE:', mean_squared_error(y_train, ridge_model.predict(X_train)))


#### Task 3.2: LASSO

Ridge regression adds a penalty term to the usual linear regression objective using $\beta \| w \|_2^2$ as the regularizer. **LASSO** implements the same intuitive idea, namely penalizing large $w$s, but it uses a different regularizer, namely $\beta \| w \|_1$.

Unfortunately, the 1-norm, which is based on the absolute value function, is not everywhere differentiable, so we cannot resort to calculus as usual to solve for a closed-form solution to LASSO. But not to fear, there are other tools in our toolbox!

Remember our good friend gradient descent. (Chef Chicken remembers her well!) Well, of course gradient descent also relies on a gradient, but it can be generalized to an algorithm called *subgradient* descent, which is slower, but still effective at minimizing non-differentiable loss functions, such as the LASSO loss function.

To build a LASSO regression model, you will use `Lasso` in `sklearn.linear_model`.

> Implement `make_lasso(beta)` so that it returns a `Lasso` object with `alpha=beta`, `max_iter=100000`, and `tol=1e-4`. Then run the pipeline and fit the degree-1 model. Record the training MSE and how many of its feature coefficients are exactly zero.

<details><summary><b>Hint: Regularization</b></summary>

Note that `scikit-learn` calls the regularization strength `alpha`. Unlike in our implementation of ridge regression, `scikit-learn` does not penalize the intercept. It also scales its loss differently, so the same value of $\beta$ won't regularize the two models by the same amount. That's why we tune them separately.

</details>

In [ ]:
def make_lasso(beta):
    # TODO
    pass

lasso_model = make_regression_pipeline(make_lasso(beta=0.1))
lasso_model.fit(X_train, y_train)
print('LASSO training MSE:', mean_squared_error(y_train, lasso_model.predict(X_train)))
for name, model in [('Ridge', ridge_model), ('LASSO', lasso_model)]:
    regressor = model.named_steps['regressor']
    coefficients = regressor.coef_ if name == 'LASSO' else regressor.get_coefficients()[1:]
    print(f'{name}: {np.sum(np.asarray(coefficients) == 0)} zero feature coefficients')

# Coefficients of the standardized features, side by side (+ 0.0 prints -0.0 as 0.0)
print((pd.DataFrame({
    'Ridge': ridge_model.named_steps['regressor'].get_coefficients()[1:],
    'LASSO': lasso_model.named_steps['regressor'].coef_,
}, index=feature_names).round(3) + 0.0).to_string())

#### Task 3.3: Explain Regularization

> Answer the following questions in your README:
>
> LASSO and ridge regression both penalize large coefficients. Compare the training errors and coefficients of the two degree-1 models. How do the differences reflect their different penalties? And why isn't a lower training MSE enough to tell you which model will generalize better?

## Part 4: Cross-Validation for Variables, Degree, and $\beta$

In this part, you will build candidate models, and use cross validation (CV) to compare candidate models trained on `X_train` and `y_train`.

To keep the computation manageable, we propose that you build your candidate models in stages, settling one choice before moving on to the next. Each stage should pick the best among the candidates it tries, according to the validation MSE. Here are the modeling choices:

1. A subset of the original life-expectancy variables.
2. A polynomial degree for those variables.
3. A regularization strength $\beta$.

Each choice of 1, 2, and 3 corresponds to a candidate. You should evaluate each candidate using five-fold CV: The training set is shuffled (`random_state=42`) and split into five roughly equal groups. Each candidate is then fit five times, each time training on four of the groups, and outputting the **validation MSE** on the remaining one. Its score is then the average of those five validation MSEs. As every candidate uses the same five groups, their scores can be compared directly.

<details><summary><b>Hint: Polynomial Regression</b></summary>

The pipeline's `PolynomialFeatures` step handles the polynomial expansion: degree 1 uses the selected features as they are, and higher degrees add powers and interaction terms. You don't need to write a separate polynomial regression.

</details>


### Provided Model Factory

The `make_model` function below builds a fresh, unfitted pipeline around your `RidgeRegression`, your `make_lasso`, or `LinearRegression`. This function takes as input a degree, a value of $\beta$, and the indices of the features to use, which refer to their positions in `feature_names`.

In [ ]:
def make_model(model_name, degree, beta, feature_indices=None):
    if model_name == 'Ridge':
        regressor = RidgeRegression(beta=beta)
    elif model_name == 'LASSO':
        regressor = make_lasso(beta)
    elif model_name == 'Linear':
        regressor = LinearRegression()
    else:
        raise ValueError(f'Unknown model: {model_name}')
    return make_regression_pipeline(regressor, degree, feature_indices)


#### Task 4.1: Compute Cross-Validation Scores

> Implement `evaluate_cv`. For each combination of model, feature subset, degree, and $\beta$, your function should:
>
> - split the training data into five groups, and for each group, fit a fresh pipeline on the other four and compute the validation MSE (i.e., the MSE on the group that was left out of training)
> - return one row per candidate with the model name, feature-set name, degree, $\beta$, mean validation MSE, and the standard error.
>
> You may use `cross_val_score` with `scoring='neg_mean_squared_error'`. Note that this function returns *negative* MSEs, so you'll need to flip the sign.
>
> The **standard error** is the sample standard deviation of the five validation MSEs divided by $\sqrt{5}$. The error bars in the plots show how the validation MSE varies from group to group; they are not confidence intervals.

In [ ]:
def evaluate_cv(X, y, feature_names, feature_sets, degrees, betas,
                model_names=('Ridge', 'LASSO'), n_splits=5):
    # TODO
    pass


def select_parameters(results):
    indices = results.groupby('model')['mean_validation_mse'].idxmin()
    return results.loc[indices].reset_index(drop=True)


#### Task 4.2: Choose Variables

> Holding the degree at 1 and $\beta$ at 0.1, compare the three predefined feature sets below. You're welcome to try out your own feature sets as well, as long as you choose them using only the training data.
>
> The cell below plots each feature set's average validation MSE. In your README, identify which feature set yields the lowest validation MSE for each model. Are the differences large compared with the error bars?

Carry each model's chosen variables over to the next task.

In [ ]:
feature_sets = {
    'Schooling': ['Schooling'],
    'Health/education': ['Schooling', 'BMI', 'Adult_Mortality', 'HIV/AIDS',
                         'Income_composition_of_resources'],
    'All numeric': feature_names,
}
feature_results = evaluate_cv(X_train, y_train, feature_names, feature_sets,
                              degrees=[1], betas=[0.1])
plot_cv_errors(feature_results, 'feature_set', 'Feature selection')
print(feature_results.to_string(index=False))
selected_features = select_parameters(feature_results).set_index('model')['feature_set'].to_dict()


#### Task 4.3: Choose Polynomial Degree

> Keeping each model's chosen feature set and $\beta = 0.1$ fixed, compare polynomial degrees 1, 2, and 3. 
>
> The cell below plots the average validation MSE versus degree, for both Ridge Regression and LASSO. In your README, report which degree yields the lowest validation MSE for each model, and explain how the validation error changes as the model becomes more complex.

<!-- You may notice that a higher degree doesn't necessarily lead to a lower error. -->

Carry each model's chosen degree over to the next task.

In [ ]:
degree_results = pd.concat([
    evaluate_cv(X_train, y_train, feature_names, {set_name: feature_sets[set_name]},
                degrees=[1, 2, 3], betas=[0.1], model_names=[model_name])
    for model_name, set_name in selected_features.items()
], ignore_index=True)
plot_cv_errors(degree_results, 'degree', 'Polynomial degree at fixed beta = 0.1')
print(degree_results.to_string(index=False))
selected_degrees = select_parameters(degree_results).set_index('model')['degree'].astype(int).to_dict()


#### Task 4.4: Choose $\beta$ at the Selected Degree

> Keeping each model's chosen feature set and degree fixed, compare the nine values of $\beta$ in `np.logspace(-3, 3, 9)`. 
>
> The cell below plots validation MSE versus $\beta$, with both axes on a log scale so that the large errors at extreme values of $\beta$ don't hide the minimum. In your README, report the value of $\beta$ that yields the lowest validation MSE for each model, and describe the shape of the curve, including any elbows or flat regions. Explain why you might want to try a wider range of $\beta$ values.

In [ ]:
beta_grid = np.logspace(-3, 3, 9)
beta_results = pd.concat([
    evaluate_cv(X_train, y_train, feature_names,
                {selected_features[name]: feature_sets[selected_features[name]]},
                degrees=[degree], betas=beta_grid, model_names=[name])
    for name, degree in selected_degrees.items()
], ignore_index=True)
plot_cv_errors(beta_results, 'beta', 'Regularization strength at each selected degree')
selected = select_parameters(beta_results)
print(selected.to_string(index=False))


#### Task 4.5: Final Held-Out Comparison

> Refit each selected model on the full training set and evaluate it once on the held-out test set.
>
> The cell below compares the training and test MSE of each model with those of an unregularized, degree-1 linear model that uses *all* the numeric features. It also reports how many of each model's coefficients are exactly zero.
>
> In your README, discuss whether the models rank the same way on the test set as they did in cross validation, and whether regularization helped. 

**Do not go back and change your choices after seeing the test errors.** 

In [ ]:
final_models = {}
comparison_rows = []
candidates = [('Linear', 'All numeric', 1, 0.0, np.nan)] + [
    (row['model'], row['feature_set'], int(row['degree']), float(row['beta']),
     float(row['mean_validation_mse'])) for _, row in selected.iterrows()
]
for model_name, set_name, degree, beta, cv_mse in candidates:
    indices = [feature_names.index(name) for name in feature_sets[set_name]]
    model = make_model(model_name, degree, beta, indices)
    model.fit(X_train, y_train)
    final_models[model_name] = model
    regressor = model.named_steps['regressor']
    coefficients = (regressor.coef_ if model_name == 'LASSO'
                    else regressor.get_coefficients()[1:])
    comparison_rows.append({
        'model': model_name, 'feature_set': set_name, 'degree': degree, 'beta': beta,
        'cv_mse': cv_mse,
        'train_mse': mean_squared_error(y_train, model.predict(X_train)),
        'test_mse': mean_squared_error(y_test, model.predict(X_test)),
        'zero_feature_coefficients': int(np.sum(np.asarray(coefficients) == 0)),
        'feature_coefficients': len(coefficients),
    })
comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))


#### Task 4.6: MCQs

> **Q5.** Which strategy prevents a model from overfitting to the training data for polynomial regression?
>
> &emsp;**A.** Increasing the polynomial degree.
>
> &emsp;**B.** Removing the bias term from the model.
>
> &emsp;**C.** Training the model on the test data in addition to train data.
>
> &emsp;**D.** Penalizing large coefficients.
>
>
> **Q6.** Comparing coefficients across standard linear, Ridge, and LASSO regression on the same data, what do you observe?
>
> &emsp;**A.** LASSO produces many coefficients that are exactly 0, while Ridge coefficients are smaller than standard linear's.
>
> &emsp;**B.** Ridge produces many coefficients that are exactly 0, while LASSO coefficients are smaller than standard linear's.
>
> &emsp;**C.** Neither LASSO nor Ridge regression changes the coefficients relative to standard linear regression.
>
> &emsp;**D.** Both LASSO's and Ridge's coefficients are smaller than standard linear's and nonzero.
>
>
> **Q7.** If Ridge has a closed-form solution while LASSO must be solved iteratively using variants of gradient descent, why is LASSO still a common form of regression?
>
> &emsp;**A.** LASSO is easier to compute since it has no closed form.
>
> &emsp;**B.** LASSO always achieves lower test error than Ridge.
>
> &emsp;**C.** LASSO's penalty term produces sparse models that yield automatic feature selection.
>
> &emsp;**D.** LASSO does not penalize coefficients as much as Ridge, and therefore it never underfits.

## Testing

Run the cell below to run your implementations against the tests in `tests/`. Make sure you've run all of the implementation cells above first.

In [ ]:
from tests.regression_tests import run_all_tests
run_all_tests(globals())

### Task 5.0: Your Own Tests

Add tests of your own for regression fitting, feature selection, cross-validation scores, and $\beta$ selection in the cell below. We've included an example to get you started.

In [ ]:
df = preprocess_data(pd.read_csv('life_expectancy.csv'))
X_school, y_school, _ = extract_features(df, 'Schooling')
X_school_train, X_school_test, y_school_train, y_school_test = train_test_split(
    X_school, y_school, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_school_train, y_school_train)
predictions = model.predict(X_school_test)
np.testing.assert_allclose(model.mse(X_school_test, y_school_test), np.mean((predictions - y_school_test) ** 2))

# TODO: Add tests for your implementations here.